$$ PE(\text{position}, 2i) = \sin\bigg( \frac{ \text{position} }{10000^\frac{2i}{d_{model}}} \bigg)$$

$$ PE(\text{position}, 2i+1) = \cos\bigg( \frac{ \text{position} }{10000^\frac{2i}{d_{model}}} \bigg) $$ 

We can rewrite these as

$$
        PE(\text{position}, i) = \sin\bigg( \frac{ \text{position} }{10000^\frac{i}{d_{model}}} \bigg) \text{ when i is even} $$
 $$
         PE(\text{position}, i) = \cos\bigg( \frac{ \text{position} }{10000^\frac{i-1}{d_{model}}} \bigg) \text{ when i is odd}
         $$

In [1]:
import torch
import torch.nn as nn

max_sequence_length = 10
d_model = 6

In [3]:
even_i = torch.arange(0, d_model, 2).float()

denominator = torch.pow(10000, even_i / d_model)
# because i - 1 denominator is the same for odd and even

position = torch.arange(0, max_sequence_length).float().reshape(-1, 1)


even_pe = torch.sin(position / denominator)
odd_pe = torch.cos(position / denominator)

pe = torch.zeros(max_sequence_length, d_model)

pe[:, 0::2] = even_pe
pe[:, 1::2] = odd_pe

pe







tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
        [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999],
        [-0.2794,  0.9602,  0.2749,  0.9615,  0.0129,  0.9999],
        [ 0.6570,  0.7539,  0.3192,  0.9477,  0.0151,  0.9999],
        [ 0.9894, -0.1455,  0.3629,  0.9318,  0.0172,  0.9999],
        [ 0.4121, -0.9111,  0.4057,  0.9140,  0.0194,  0.9998]])

In [5]:
class PositionalEncoding(torch.nn.Module):

    def __init__(self, d_model: int, max_seq_len: int):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_len, d_model)
        i_arr = torch.arange(0, d_model, 2).float()
        denominator = torch.pow(10000,  i_arr / d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()

        pe[:, 0::2] = torch.sin(position / denominator)
        pe[:, 1::2] = torch.cos(position / denominator)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    
pe = PositionalEncoding(d_model, max_sequence_length)

x = torch.randn(1, max_sequence_length, d_model)
pe(x)

tensor([[[-1.3870,  1.6655, -0.9416,  1.0737,  0.4558,  2.4284],
         [ 0.0850, -1.0432,  0.0911,  1.2995,  0.0390,  0.2031],
         [ 0.9022, -0.2880, -0.6931,  1.2695,  0.4566,  1.5947],
         [ 0.5646, -1.2104, -2.2778,  1.5243,  0.0543, -0.3042],
         [-1.1652, -0.9861,  0.2670, -2.3148,  2.1800, -0.4360],
         [-3.7038,  1.0885,  0.2062,  1.5998,  1.1363,  1.6171],
         [ 0.4207,  1.3346,  0.5755,  0.6339, -0.6350,  2.8975],
         [ 1.1636, -0.6774, -0.0617,  0.9710, -0.3701, -0.0774],
         [ 2.4229, -0.3119,  0.7376,  1.9088,  0.3515,  2.4769],
         [ 0.7337, -0.2527,  0.3335,  0.4996,  1.3852,  0.0126]]])